Giai đoạn 4.1: Content-Based Filtering

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv('../data/processed/movies_features.csv')

with open('../models_artifacts/tfidf_matrix.pkl', 'rb') as f:
    tfidf_matrix = pickle.load(f)

print("movies:", movies.shape)
print("tfidf_matrix:", tfidf_matrix.shape)

movies: (45429, 9)
tfidf_matrix: (45429, 5000)


Bước 4.1.1: Xây index tra cứu phim theo tên

In [2]:
movies = movies.reset_index(drop=True)

# Lưu ý: EDA đã phát hiện 3,188 title bị trùng lặp (phim remake/cùng tên)
# -> khi tra cứu theo title, mỗi tên chỉ lấy phim đầu tiên khớp
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates(keep='first')

print("Số title duy nhất dùng được để tra cứu:", indices.shape[0])
print("Số dòng bị 'che khuất' do trùng title:", movies.shape[0] - indices.shape[0])

Số title duy nhất dùng được để tra cứu: 45429
Số dòng bị 'che khuất' do trùng title: 0


Bước 4.1.2: Hàm gợi ý Content-Based

In [3]:
def get_content_based_recommendations(title, top_n=5):
    if title not in indices:
        print(f"Không tìm thấy phim: {title}")
        return pd.DataFrame()
    
    idx = indices[title]
    if isinstance(idx, pd.Series):  # phòng trường hợp vẫn còn trùng
        idx = idx.iloc[0]
    
    # Tính similarity giữa 1 phim này với toàn bộ tập phim (không tính ma trận đầy đủ)
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_indices = sim_scores.argsort()[::-1]
    similar_indices = [i for i in similar_indices if i != idx][:top_n]
    
    result = movies.iloc[similar_indices][['id', 'title', 'weighted_rating', 'vote_average']].copy()
    result['similarity_score'] = sim_scores[similar_indices]
    return result.reset_index(drop=True)

get_content_based_recommendations('The Dark Knight', top_n=5)

,id,title,weighted_rating,vote_average,similarity_score
0,49026,The Dark Knight Rises,7.592753,7.6,0.452343
1,300424,LEGO DC Comics Super Heroes: Batman: Be-Leaguered,5.796804,6.1,0.427166
2,40662,Batman: Under the Red Hood,7.463342,7.6,0.399036
3,269246,Batman Beyond Darwyn Cooke's Batman 75th Anniv...,6.161465,7.7,0.384484
4,364,Batman Returns,6.580820,6.6,0.377649


Bước 4.1.3: Kiểm định định tính với vài phim khác

In [4]:
for t in ['Toy Story', 'The Godfather', 'Inception', 'La La Land']:
    print(f"--- Vì bạn thích: {t} ---")
    res = get_content_based_recommendations(t, top_n=5)
    if not res.empty:
        print(res[['title', 'similarity_score']].to_string(index=False))
    print()

--- Vì bạn thích: Toy Story ---
                           title  similarity_score
                     Toy Story 3          0.466580
          The 40 Year Old Virgin          0.414782
                       The Champ          0.398018
                     Toy Story 2          0.391404
Andy Kaufman Plays Carnegie Hall          0.388514

--- Vì bạn thích: The Godfather ---
                      title  similarity_score
     Mörderische Erpressung          0.272084
Between Eleven and Midnight          0.272084
                      Petos          0.272084
Un milliard dans un billard          0.272084
  Barrela: Escola de Crimes          0.272084

--- Vì bạn thích: Inception ---
                       title  similarity_score
          My Night, Your Day          0.211253
                      Cypher          0.198437
               Spy, Stand Up          0.193006
              Straight Story          0.187487
UFO - Distruggete base Luna!          0.182658

--- Vì bạn thích: La La Land ---


Bước 4.1.4: Đo thời gian chạy

In [5]:
import time

start = time.time()
_ = get_content_based_recommendations('Inception', top_n=5)
elapsed = time.time() - start
print(f"Thời gian tính 1 lần gợi ý: {elapsed:.3f} giây")

Thời gian tính 1 lần gợi ý: 0.081 giây


In [6]:
# Chẩn đoán vấn đề 1: xem soup và vote_count của các phim "khả nghi"
suspects = ['Mörderische Erpressung', 'Between Eleven and Midnight', 'Petos', 
            'Un milliard dans un billard', 'Barrela: Escola de Crimes']

for t in suspects:
    if t in indices:
        idx = indices[t]
        if isinstance(idx, pd.Series):
            idx = idx.iloc[0]
        row = movies.iloc[idx]
        print(f"{t}")
        print(f"  soup: '{row['soup']}'")
        print(f"  vote_count: {row['vote_count']}, popularity: {row['popularity']}")
        print()

Mörderische Erpressung
  soup: 'crime hinnerkschönemann mirabartuschek tobiasschenke markusimboden markusimboden'
  vote_count: 0.0, popularity: 0.003565

Between Eleven and Midnight
  soup: 'crime louisjouvet madeleinerobinson robertarnoux henridecoin henridecoin'
  vote_count: 0.0, popularity: 0.001443

Petos
  soup: 'crime ryöstö poliisi gangsteri vankila perustuunäytelmään häät itsemurha pako perustuutositapahtumiin paavopentikäinen eevalitmanen marttitschokkinen taavikassila taavikassila'
  vote_count: 1.0, popularity: 0.03795

Un milliard dans un billard
  soup: 'crime jeanseberg clauderich elsamartinelli nicolasgessner nicolasgessner'
  vote_count: 0.0, popularity: 0.002612

Barrela: Escola de Crimes
  soup: 'crime prisoncell malerape paulocésarperéio marcospalmeira cláudiomamberti marcoantoniocury marcoantoniocury'
  vote_count: 0.0, popularity: 0.008848



In [7]:
# So sánh với soup của chính The Godfather
idx_godfather = indices['The Godfather']
if isinstance(idx_godfather, pd.Series):
    idx_godfather = idx_godfather.iloc[0]
print("The Godfather soup:", movies.iloc[idx_godfather]['soup'])
print("vote_count:", movies.iloc[idx_godfather]['vote_count'])

The Godfather soup: drama crime italy loveatfirstsight lossoffather patriarch organizedcrime mafia lawyer italianamerican crimefamily risetopower mobboss 1940s marlonbrando alpacino jamescaan francisfordcoppola francisfordcoppola spanning the years 1945 to 1955, a chronicle of the fictional italian-american corleone crime family. when organized crime family patriarch, vito corleone barely survives an attempt on his life, his youngest son, michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.
vote_count: 6024.0


In [8]:
# Kiểm tra phân bố độ dài soup và mối liên hệ với vote_count
movies['soup_len'] = movies['soup'].str.split().str.len()
print(movies['soup_len'].describe())

# Bao nhiêu phim có soup rất ngắn (<=3 từ) VÀ vote_count rất thấp (<=1)?
very_sparse = movies[(movies['soup_len'] <= 3) & (movies['vote_count'] <= 1)]
print("Số phim soup ngắn + gần như không ai vote:", very_sparse.shape[0])

count    45429.000000
mean        64.125096
std         35.756547
min          0.000000
25%         36.000000
50%         58.000000
75%         84.000000
max        326.000000
Name: soup_len, dtype: float64
Số phim soup ngắn + gần như không ai vote: 64


In [9]:
# Xác định candidate pool: chỉ phim có đủ dữ liệu tin cậy mới được đề xuất
# (phim vẫn có thể được dùng làm "phim nguồn" để tìm gợi ý, chỉ giới hạn ở phía kết quả)
MIN_VOTE_COUNT = 10  # ngưỡng tối thiểu để loại phim gần như không ai biết

candidate_mask = movies['vote_count'] >= MIN_VOTE_COUNT
print(f"Số phim đủ điều kiện làm candidate: {candidate_mask.sum()} / {len(movies)} ({candidate_mask.mean():.1%})")

candidate_indices = movies.index[candidate_mask].to_numpy()

Số phim đủ điều kiện làm candidate: 22915 / 45429 (50.4%)


In [10]:
def get_content_based_recommendations_v2(title, top_n=5):
    if title not in indices:
        print(f"Không tìm thấy phim: {title}")
        return pd.DataFrame()
    
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    
    # Chỉ tính similarity với các candidate đủ điều kiện (nhanh hơn + sạch hơn)
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix[candidate_indices]).flatten()
    
    order = sim_scores.argsort()[::-1]
    top_candidate_positions = order[:top_n + 1]  # +1 phòng trường hợp trùng chính nó
    
    result_indices = candidate_indices[top_candidate_positions]
    result_scores = sim_scores[top_candidate_positions]
    
    # Loại chính phim nguồn nếu lọt vào (trường hợp phim nguồn cũng nằm trong candidate pool)
    mask_self = result_indices != idx
    result_indices = result_indices[mask_self][:top_n]
    result_scores = result_scores[mask_self][:top_n]
    
    result = movies.iloc[result_indices][['id', 'title', 'weighted_rating', 'vote_average']].copy()
    result['similarity_score'] = result_scores
    return result.reset_index(drop=True)

In [11]:
# Kiểm định lại với đúng 4 phim đã bị lỗi trước đó
for t in ['Toy Story', 'The Godfather', 'Inception', 'La La Land']:
    print(f"--- Vì bạn thích: {t} ---")
    res = get_content_based_recommendations_v2(t, top_n=5)
    if not res.empty:
        print(res[['title', 'similarity_score']].to_string(index=False))
    print()

--- Vì bạn thích: Toy Story ---
                 title  similarity_score
           Toy Story 3          0.466580
The 40 Year Old Virgin          0.414782
             The Champ          0.398018
           Toy Story 2          0.391404
         A Simple Life          0.263308

--- Vì bạn thích: The Godfather ---
                  title  similarity_score
 The Godfather: Part II          0.259641
The Godfather: Part III          0.242761
   Jane Austen's Mafia!          0.229960
          Live by Night          0.225845
          Henry's Crime          0.219260

--- Vì bạn thích: Inception ---
          title  similarity_score
         Cypher          0.198437
 Blood and Wine          0.173575
       Altitude          0.171123
         Duplex          0.169521
Pitch Perfect 2          0.165230

--- Vì bạn thích: La La Land ---
                           title  similarity_score
                            Nina          0.311705
                 Born to Be Blue          0.263826
Guy and M

In [12]:
# Đo lại thời gian chạy (kỳ vọng nhanh hơn vì ma trận candidate nhỏ hơn)
start = time.time()
_ = get_content_based_recommendations_v2('Inception', top_n=5)
print(f"Thời gian: {time.time() - start:.3f} giây")

Thời gian: 0.046 giây


In [13]:
import pickle

# Lưu candidate pool + indices để dùng lại trong app Streamlit
with open('../models_artifacts/content_based_candidate_indices.pkl', 'wb') as f:
    pickle.dump(candidate_indices, f)

with open('../models_artifacts/title_indices.pkl', 'wb') as f:
    pickle.dump(indices, f)

print("Đã lưu artifact Content-Based")

Đã lưu artifact Content-Based
